In [ ]:
import os
import sys
notebook_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(notebook_dir, '../..')) # Adjust as needed
if project_root not in sys.path:
    sys.path.append(project_root) # add notebook to sys.path

In [ ]:
import torch

In [ ]:
device = torch.device('cpu') if not torch.accelerator.is_available() else torch.accelerator.current_accelerator()
print(device)

# Hyperparameters

In [ ]:
batch_size = 128
resize = (224, 224)
num_epochs = 20
lr = 0.001

# Load dataset

In [ ]:
from utils.data import CIFAR10
from utils.train import train_model, val_stats


def get_optimizer(model):
    return torch.optim.Adam(model.parameters(), lr=lr)

def fit(model, num_epochs=num_epochs):
    dataset = CIFAR10(batch_size=batch_size, resize=resize, device=device)
    train_dl, val_dl = dataset.get_dataloaders()
    optimizer = get_optimizer(model)
    return train_model(model, train_dl, val_dl, optimizer, num_epochs)

# Create model

We'll be implementing the Network in Network architecture here.

What makes the NiN different is (a) the lack of linear layers at the end, (b) introducing 1x1 convolutions to introduce local non-linearities between the features, and (c) to use a global pooling layer at the end. 

Note that the global pooling layer (c) would not be possible without the 1x1 convolutions (b).

In [ ]:
from torch import nn
from utils.models import Module

In [ ]:
def nin_block(out_channels, kernel_size, stride, padding):
    return nn.Sequential(
        nn.LazyConv2d(out_channels, kernel_size, stride=stride, padding=padding), nn.ReLU(),
        nn.LazyConv2d(out_channels, kernel_size=1), nn.ReLU(), 
        nn.LazyConv2d(out_channels, kernel_size=1), nn.ReLU()
    )

In [ ]:
class NiN(Module):
    def __init__(self, dropout=0.5, num_classes=10):
        super().__init__()
        self.dropout = dropout
        self.num_classes = num_classes
        self.net = nn.Sequential(
            nin_block(96, kernel_size=11, stride=4, padding=0),
            nn.MaxPool2d(kernel_size=3, stride=2),
            nin_block(256, kernel_size=5, stride=1, padding=2),
            nn.MaxPool2d(kernel_size=3, stride=2),
            nin_block(384, kernel_size=3, stride=1, padding=1),
            nn.MaxPool2d(kernel_size=3, stride=2),
            nn.Dropout(p=self.dropout),
            nin_block(num_classes, kernel_size=3, stride=1, padding=1),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten()
        )
    
    def forward(self, X):
        return self.net(X)

In [ ]:
model = NiN().to(device)
X = torch.randn(batch_size, 3, 224, 224).to(device)

In [ ]:
with torch.no_grad():
    print(model(X).shape)

print(model)

In [ ]:
num_params = sum(p.numel() for p in model.parameters())
print(num_params)
print(f"Parameter memory: {num_params * 4 / 1e6:.0f} MB")

# Train model

In [ ]:
fit(model);

In [ ]:
_, val_dl = CIFAR10(batch_size=batch_size, resize=resize, device=device).get_dataloaders()
val_loss, val_accuracy = val_stats(model, val_dl, device=device)

In [ ]:
print(f"Validation loss: {val_loss:.2f}, accuracy: {val_accuracy:.2f}%")

# How much memory usage during training?

In [ ]:
print(f"Max memory allocated: {torch.cuda.max_memory_allocated()/1e9:.3f} GB")